# Binary Classification with Neural Networks on the Census Income Dataset

Predict whether a person earns more than $50,000 annually using PyTorch.

**Dataset:** [income.csv on Google Drive](https://drive.google.com/file/d/1ay5vOv2YiOwjKIWnXFT6sptW0gqew0oa/view?usp=sharing)

This notebook separates categorical features, continuous features, and the target label; creates a 25,000-row training set and 5,000-row test set; trains a neural network for 300 epochs; and reports test loss and accuracy.

## 1. Install and import dependencies
Run `pip install -r requirements.txt` before running the notebook.

In [1]:
import os
import random
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


## 2. Load the dataset
Place `income.csv` in the project root or `notebooks/` folder. If it is not found, the cell attempts to download the shared Google Drive file.

In [2]:
DATA_PATHS = ["income.csv", "../income.csv", "notebooks/income.csv"]
DATA_PATH = next((p for p in DATA_PATHS if os.path.exists(p)), None)

if DATA_PATH is None:
    # Requires the Google Drive file to be shared publicly.
    file_id = "1ay5vOv2YiOwjKIWnXFT6sptW0gqew0oa"
    DATA_PATH = "income.csv"
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    try:
        urllib.request.urlretrieve(url, DATA_PATH)
    except Exception as exc:
        raise FileNotFoundError(
            "Could not download income.csv automatically. Download it from the dataset link "
            "in the README and place it in the project root."
        ) from exc

df = pd.read_csv(DATA_PATH, skipinitialspace=True)
df.columns = [str(c).strip().lower().replace("-", "_").replace(" ", "_") for c in df.columns]
print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

Dataset shape: (30000, 10)


,age,sex,education,education_num,marital_status,workclass,occupation,hours_per_week,income,label
0,27,Male,HS-grad,9,Never-married,Private,Craft-repair,40,<=50K,0
1,47,Male,Masters,14,Married,Local-gov,Exec-managerial,50,>50K,1
2,59,Male,HS-grad,9,Divorced,Self-emp,Prof-specialty,20,<=50K,0
3,38,Female,Prof-school,15,Never-married,Federal-gov,Prof-specialty,57,>50K,1
4,64,Female,11th,7,Widowed,Private,Farming-fishing,40,<=50K,0


Columns: ['age', 'sex', 'education', 'education_num', 'marital_status', 'workclass', 'occupation', 'hours_per_week', 'income', 'label']


## 3. Prepare categorical, continuous, and label columns
The target column is detected from common Census Income column names. Categorical features are ordinal-encoded for embeddings; continuous features are standardized. Missing/unknown category values are handled safely.

In [3]:
target_candidates = ["income", "income_bracket", "class", "label", "salary", "annual_income"]
target_col = next((c for c in target_candidates if c in df.columns), None)
if target_col is None:
    # Common alternative: target values are stored in the final column.
    target_col = df.columns[-1]
    print(f"Target column not found by name; using final column: {target_col}")

df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(str).str.strip().str.rstrip(".")
# Common Census Income labels: <=50K and >50K (sometimes with spaces or periods).
def encode_income(value):
    v = str(value).strip().lower().rstrip(".").replace(" ", "")
    if v in {">50k", "1", "high", "yes", "true"} or " >50k" in v:
        return 1
    if v in {"<=50k", "≤50k", "0", "low", "no", "false"}:
        return 0
    # Handle labels that contain a threshold phrase.
    if ">50k" in v:
        return 1
    if "<=50k" in v or "≤50k" in v:
        return 0
    return np.nan

y_series = df[target_col].map(encode_income)
if y_series.isna().any():
    unique_values = df[target_col].unique().tolist()
    raise ValueError(f"Could not map all target labels in '{target_col}'. Values: {unique_values[:20]}")
y = y_series.astype("int64").to_numpy()

X = df.drop(columns=[target_col]).copy()
# Treat text/object/category/boolean columns as categorical.
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
continuous_cols = [c for c in X.columns if c not in categorical_cols]

# Convert any remaining non-numeric columns to categorical, and numeric columns to numbers.
for col in continuous_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
for col in categorical_cols:
    X[col] = X[col].fillna("Unknown").astype(str).str.strip().replace("", "Unknown")

X[continuous_cols] = X[continuous_cols].replace([np.inf, -np.inf], np.nan)
for col in continuous_cols:
    X[col] = X[col].fillna(X[col].median())

print("Target column:", target_col)
print("Categorical columns:", categorical_cols)
print("Continuous columns:", continuous_cols)
print("Label distribution (0 = <=50K, 1 = >50K):")
print(pd.Series(y).value_counts().sort_index())

Target column: income
Categorical columns: ['sex', 'education', 'marital_status', 'workclass', 'occupation']
Continuous columns: ['age', 'education_num', 'hours_per_week', 'label']
Label distribution (0 = <=50K, 1 = >50K):
0    21700
1     8300
Name: count, dtype: int64


## 4. Split into 25,000 training rows and 5,000 testing rows
The assignment specifies 30,000 records. This cell checks the dataset size before splitting.

In [4]:
if len(X) != 30000:
    print(f"Note: dataset contains {len(X):,} usable rows rather than exactly 30,000.")
if len(X) < 30000:
    raise ValueError("The assignment requires 30,000 usable rows (25,000 train and 5,000 test). Check income.csv.")
if len(X) > 30000:
    # Use 30,000 rows to match the assignment's requested split.
    X = X.iloc[:30000].copy()
    y = y[:30000]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=25000, test_size=5000, random_state=SEED, stratify=y
)

# Fit preprocessing on training data only to avoid data leakage.
if categorical_cols:
    cat_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1,
        encoded_missing_value=-1, dtype=np.int64
    )
    cat_train = cat_encoder.fit_transform(X_train[categorical_cols]).astype(np.int64) + 1
    cat_test = cat_encoder.transform(X_test[categorical_cols]).astype(np.int64) + 1
    cat_test = np.maximum(cat_test, 0)
    # Index 0 is reserved for unseen categories; known categories begin at 1.
    cat_cardinalities = [len(cats) + 1 for cats in cat_encoder.categories_]
else:
    cat_train = np.empty((len(X_train), 0), dtype=np.int64)
    cat_test = np.empty((len(X_test), 0), dtype=np.int64)
    cat_cardinalities = []

if continuous_cols:
    scaler = StandardScaler()
    cont_train = scaler.fit_transform(X_train[continuous_cols]).astype(np.float32)
    cont_test = scaler.transform(X_test[continuous_cols]).astype(np.float32)
else:
    cont_train = np.empty((len(X_train), 0), dtype=np.float32)
    cont_test = np.empty((len(X_test), 0), dtype=np.float32)

X_cat_train = torch.tensor(cat_train, dtype=torch.long)
X_cat_test = torch.tensor(cat_test, dtype=torch.long)
X_cont_train = torch.tensor(cont_train, dtype=torch.float32)
X_cont_test = torch.tensor(cont_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("Training rows:", len(y_train_tensor))
print("Testing rows:", len(y_test_tensor))
print("Categorical tensor shape:", tuple(X_cat_train.shape))
print("Continuous tensor shape:", tuple(X_cont_train.shape))
print("Label tensor shape:", tuple(y_train_tensor.shape))

Training rows: 25000
Testing rows: 5000
Categorical tensor shape: (25000, 5)
Continuous tensor shape: (25000, 4)
Label tensor shape: (25000,)


## 5. Define the TabularModel
Categorical features are represented using learned embeddings. Continuous features pass through batch normalization. The network has one hidden layer with 50 neurons and dropout probability 0.4. Two output logits are used with `CrossEntropyLoss`.

In [5]:
class TabularModel(nn.Module):
    def __init__(self, cat_cardinalities, n_continuous, hidden_size=50, dropout_p=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, min(50, (cardinality + 1) // 2))
            for cardinality in cat_cardinalities
        ])
        embedding_size = sum(emb.embedding_dim for emb in self.embeddings)
        self.batch_norm = nn.BatchNorm1d(n_continuous) if n_continuous > 0 else None
        input_size = embedding_size + n_continuous
        if input_size == 0:
            raise ValueError("No input features were detected.")
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_p)
        self.output = nn.Linear(hidden_size, 2)

    def forward(self, x_cat, x_cont):
        parts = []
        if len(self.embeddings) > 0:
            embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            parts.append(torch.cat(embedded, dim=1))
        if self.batch_norm is not None:
            parts.append(self.batch_norm(x_cont))
        x = torch.cat(parts, dim=1)
        x = self.dropout(self.activation(self.hidden(x)))
        return self.output(x)

model = TabularModel(
    cat_cardinalities=cat_cardinalities,
    n_continuous=len(continuous_cols),
    hidden_size=50,
    dropout_p=0.4
).to(DEVICE)
print(model)

TabularModel(
  (embeddings): ModuleList(
    (0): Embedding(3, 2)
    (1): Embedding(15, 8)
    (2): Embedding(7, 4)
    (3): Embedding(6, 3)
    (4): Embedding(13, 7)
  )
  (batch_norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (hidden): Linear(in_features=28, out_features=50, bias=True)
  (activation): ReLU()
  (dropout): Dropout(p=0.4, inplace=False)
  (output): Linear(in_features=50, out_features=2, bias=True)
)


## 6. Train for 300 epochs
The model uses `CrossEntropyLoss` and Adam with learning rate 0.001. A mini-batch DataLoader makes training more memory-efficient.

In [6]:
BATCH_SIZE = 512
train_dataset = TensorDataset(X_cat_train, X_cont_train, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 300
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    seen = 0
    for batch_cat, batch_cont, batch_y in train_loader:
        batch_cat = batch_cat.to(DEVICE)
        batch_cont = batch_cont.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(batch_cat, batch_cont)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(batch_y)
        seen += len(batch_y)

    epoch_loss = running_loss / seen
    history.append(epoch_loss)
    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:03d}/{EPOCHS} | Training loss: {epoch_loss:.4f}")

Epoch 001/300 | Training loss: 0.5231
Epoch 025/300 | Training loss: 0.0006
Epoch 050/300 | Training loss: 0.0001
Epoch 075/300 | Training loss: 0.0001
Epoch 100/300 | Training loss: 0.0000
Epoch 125/300 | Training loss: 0.0000
Epoch 150/300 | Training loss: 0.0000
Epoch 175/300 | Training loss: 0.0000
Epoch 200/300 | Training loss: 0.0000
Epoch 225/300 | Training loss: 0.0000
Epoch 250/300 | Training loss: 0.0000
Epoch 275/300 | Training loss: 0.0000
Epoch 300/300 | Training loss: 0.0000


## 7. Evaluate on the test set
The reported test loss is cross-entropy loss, and accuracy is the percentage of correct predictions.

In [7]:
model.eval()
with torch.no_grad():
    test_logits = model(X_cat_test.to(DEVICE), X_cont_test.to(DEVICE))
    test_loss = criterion(test_logits, y_test_tensor.to(DEVICE)).item()
    test_predictions = torch.argmax(test_logits, dim=1).cpu().numpy()

test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print("\nClassification report:")
print(classification_report(
    y_test, test_predictions,
    labels=[0, 1],
    target_names=["<=50K", ">50K"],
    zero_division=0
))

Test Loss: 0.0000
Test Accuracy: 100.00%

Classification report:
              precision    recall  f1-score   support

       <=50K       1.00      1.00      1.00      3617
        >50K       1.00      1.00      1.00      1383

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



## 8. Optional: predict for new data
Enter values for selected features. Unspecified features use the training-set median (continuous) or `Unknown` (categorical). The function below expects a dictionary using the original column names, such as `{'marital_status': 'Married-civ-spouse', 'education': 'Bachelors', 'hours_per_week': 40}`.

In [8]:
def predict_income(user_values):
    row = {}
    for col in categorical_cols:
        row[col] = str(user_values.get(col, "Unknown"))
    for col in continuous_cols:
        value = user_values.get(col, X_train[col].median())
        row[col] = pd.to_numeric(value, errors="coerce")
        if pd.isna(row[col]):
            row[col] = X_train[col].median()

    new_df = pd.DataFrame([row], columns=X.columns)
    if categorical_cols:
        new_cat = cat_encoder.transform(new_df[categorical_cols]).astype(np.int64) + 1
        new_cat = np.maximum(new_cat, 0)
    else:
        new_cat = np.empty((1, 0), dtype=np.int64)
    if continuous_cols:
        new_cont = scaler.transform(new_df[continuous_cols]).astype(np.float32)
    else:
        new_cont = np.empty((1, 0), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        logits = model(
            torch.tensor(new_cat, dtype=torch.long, device=DEVICE),
            torch.tensor(new_cont, dtype=torch.float32, device=DEVICE)
        )
        probabilities = torch.softmax(logits, dim=1)[0]
        predicted_class = int(torch.argmax(probabilities).item())

    return {
        "prediction": ">50K" if predicted_class == 1 else "<=50K",
        "probability_<=50K": round(float(probabilities[0].item()), 4),
        "probability_>50K": round(float(probabilities[1].item()), 4)
    }

# Example: adapt keys and values to the columns in income.csv.
# print(predict_income({"education": "Bachelors", "marital_status": "Married-civ-spouse", "hours_per_week": 40}))

## Results

Run all cells from top to bottom to generate the actual test loss, test accuracy, and classification report. Results can vary slightly with the dataset and runtime environment.

## Notes
- Do not commit private or restricted datasets to a public repository. If the dataset is not permitted to be shared, keep it local and provide the authorized dataset link in the README.
- The preprocessing encoders and scaler are fit on training data only.